# Predictive Model: Claim Denial Risk

This notebook builds a simple classification model to predict whether a claim 
will be denied, using the features identified as significant during EDA 
(prior authorization status, documentation completeness).

## Steps
1. Load data
2. Feature selection & target creation
3. Encode categorical features
4. Train/test split
5. Train model
6. Evaluate model
7. Feature importance

In [1]:
import pandas as pd
df = pd.read_csv('claims_main.csv')

In [2]:
df.shape

(120000, 25)

## 1. Feature Selection & Target Creation
Simplifying the 4-category outcome into a binary target: denied (1) vs not denied (0). 
Selecting prior_auth_obtained, documentation_completeness, payer_type, and 
provider_specialty as model features, based on EDA findings. Excluding 
denial_reason_code and denial_category to avoid data leakage, since these 
fields only exist after a claim has already been denied.

In [3]:
df['is_denied'] = (df['outcome'] == 'denied').astype(int)
df['is_denied'].value_counts()

is_denied
0    86336
1    33664
Name: count, dtype: int64

In [4]:
model_df = df[['prior_auth_obtained', 'documentation_completeness', 'payer_type', 'provider_specialty', 'is_denied']]
model_df.head()

,prior_auth_obtained,documentation_completeness,payer_type,provider_specialty,is_denied
0,NaN,0.40,Commercial_PPO,Emergency_Medicine,1
1,NaN,0.37,Medicare_FFS,Internal_Medicine,1
2,NaN,0.65,Commercial_EPO,Cardiology,1
3,NaN,0.85,Commercial_PPO,Physical_Therapy,0
4,False,0.38,Medicaid_Managed,Obstetrics_Gynecology,1


### Scoping the model to claims requiring prior authorization

`prior_auth_obtained` is blank (NaN) for claims that never required prior authorization 
in the first place, this is different from claims that required it but failed to obtain 
it. Mixing these two cases would confuse the model, since a blank value doesn't mean 
the same thing as "not obtained".

To keep the model's signal clean, this model is scoped specifically to claims that 
required prior authorization, answering the question: "given a claim that needs 
prior auth, what's its risk of denial?"

In [5]:
model_df = df[df['prior_auth_required'] == True][['prior_auth_obtained', 'documentation_completeness', 'payer_type', 'provider_specialty', 'is_denied']]
model_df.shape

(54172, 5)

In [6]:
model_df.head()

,prior_auth_obtained,documentation_completeness,payer_type,provider_specialty,is_denied
4,False,0.38,Medicaid_Managed,Obstetrics_Gynecology,1
5,False,0.39,Medicare_Advantage,Primary_Care,1
8,False,0.68,Medicare_FFS,Neurology,1
9,True,0.40,Commercial_HMO,Cardiology,1
10,True,0.85,Medicare_Advantage,Oncology,0


### One-Hot Encoding Categorical Features

`payer_type` and `provider_specialty` are text categories, but models require 
numeric input. One-hot encoding converts each category into its own binary 
(1/0) column, avoiding the false assumption of ranking or order that would 
occur if categories were simply assigned numbers (e.g. 1, 2, 3).

Using pandas' `get_dummies()` to automatically create these binary columns 
for both categorical features.

In [7]:
model_df_encoded = pd.get_dummies(model_df, columns=['payer_type', 'provider_specialty'])
model_df_encoded.head()

,prior_auth_obtained,documentation_completeness,is_denied,payer_type_Commercial_EPO,payer_type_Commercial_HMO,payer_type_Commercial_PPO,payer_type_Medicaid_Managed,payer_type_Medicare_Advantage,payer_type_Medicare_FFS,provider_specialty_Behavioral_Health,...,provider_specialty_Gastroenterology,provider_specialty_Internal_Medicine,provider_specialty_Neurology,provider_specialty_Obstetrics_Gynecology,provider_specialty_Oncology,provider_specialty_Orthopedic_Surgery,provider_specialty_Physical_Therapy,provider_specialty_Primary_Care,provider_specialty_Pulmonology,provider_specialty_Radiology
4,False,0.38,1,False,False,False,True,False,False,False,...,False,False,False,True,False,False,False,False,False,False
5,False,0.39,1,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,True,False,False
8,False,0.68,1,False,False,False,False,False,True,False,...,False,False,True,False,False,False,False,False,False,False
9,True,0.40,1,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
10,True,0.85,0,False,False,False,False,True,False,False,...,False,False,False,False,True,False,False,False,False,False


### Splitting Features and Target, then Train/Test Split

Separating the encoded table into two pieces: X (all feature columns the model 
will learn from) and y (just the is_denied target column, the answer we're 
trying to predict).

Then splitting into training and testing sets (80/20) using scikit-learn's 
train_test_split, the model will learn only from the training set, and get 
tested afterwards on the unseen test set, to check if it actually generalizes 
rather than just memorizing.

In [10]:
from sklearn.model_selection import train_test_split

X = model_df_encoded.drop('is_denied', axis=1)
y = model_df_encoded['is_denied']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
X_train.shape, X_test.shape

((43337, 21), (10835, 21))

### Training the Model

Using Logistic Regression, a standard model for binary (yes/no) classification 
problems. The model learns the relationship between the features and the 
is_denied target using the training data only.

In [12]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


### Evaluating the Model

Testing the trained model on the held-out test set (claims it has never seen 
during training) to check how well it generalizes. Using accuracy as the 
first evaluation metric, the percentage of test claims where the model's 
prediction matched the actual outcome.

In [13]:
y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2%}")

Accuracy: 91.46%


### Model Results

Achieved 91.46% accuracy on the held-out test set. This strong result aligns 
with EDA findings, documentation completeness alone showed a near-perfect 
relationship with denial (0% denied for Very High completeness, 88% for Low), 
giving the model a genuinely strong signal to learn from.

Note: accuracy alone can be misleading if outcomes are imbalanced. Next, 
checking feature importance and additional metrics to confirm the model isn't 
just favoring the majority class.

In [14]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[7301  425]
 [ 500 2609]]
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      7726
           1       0.86      0.84      0.85      3109

    accuracy                           0.91     10835
   macro avg       0.90      0.89      0.89     10835
weighted avg       0.91      0.91      0.91     10835



### Confusion Matrix & Classification Report

Confusion matrix confirms the model is genuinely learning, not just guessing 
the majority class: it correctly identified 2,609 out of 3,109 actual denied 
claims (84% recall), and 7,301 out of 7,726 actual non-denied claims (94% recall).

Precision for denied claims is 0.86, meaning when the model predicts "denied," 
it's correct 86% of the time. This confirms the 91.46% accuracy is a genuine, 
balanced result and not inflated by class imbalance.

### Feature Importance

Checking which features the model relied on most, using the coefficients from 
the trained Logistic Regression model. Larger coefficients (positive or 
negative) indicate stronger influence on the denial prediction.

Expecting documentation_completeness and prior_auth_obtained to show the 
strongest influence, mathematically confirming the findings from the EDA phase.

In [15]:
importance = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': model.coef_[0]
})
importance = importance.sort_values('coefficient', ascending=False)
importance

,feature,coefficient
5,payer_type_Medicaid_Managed,1.644094
4,payer_type_Commercial_PPO,1.617907
6,payer_type_Medicare_Advantage,1.600979
3,payer_type_Commercial_HMO,1.557393
7,payer_type_Medicare_FFS,1.526067
2,payer_type_Commercial_EPO,1.484710
15,provider_specialty_Oncology,0.857265
11,provider_specialty_Gastroenterology,0.834076
19,provider_specialty_Pulmonology,0.785231
10,provider_specialty_Emergency_Medicine,0.773562


## Model Summary

Built a Logistic Regression model to predict claim denial risk for claims 
requiring prior authorization, using documentation_completeness, 
prior_auth_obtained, payer_type, and provider_specialty as features.

**Results:**
- 91.46% accuracy on held-out test data
- Correctly identified 84% of actual denied claims (recall), 86% precision
- documentation_completeness emerged as the single strongest predictor 
  (coefficient of -19.28, far larger than any other feature), confirming 
  it as the dominant driver of denial risk found during EDA
- prior_auth_obtained was also a meaningful negative predictor, confirming 
  the earlier finding that obtaining authorization reduces denial risk
- payer_type and provider_specialty showed relatively uniform, smaller 
  coefficients, consistent with the EDA finding that payer alone doesn't 
  meaningfully drive denial risk

This model mathematically validates the two actionable findings from EDA documentation completeness and prior authorization status as genuine, strong predictors of claim denial, not coincidental patterns.